In [0]:
import configparser
from pathlib import Path

In [0]:
%run "../helpers/000-log-helper"

In [0]:
# Get the name of the notebook
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_name = nb_path.split("/")[-1]

# Instantiate the helper class to get the Spark session and logger
helper = HELPER()
spark = helper.get_spark_session(nb_name)
logger = helper.get_logger(nb_name)


In [0]:
# Read the config file
config_path_file = "/Workspace/Repos/development/datavault-datamodelling/config/schema.config"

notebook_path = "./150-create-schema"

In [0]:
# function to read schemas
def load_schemas_config(config_path: str):
    """
    Reads the config file and returns a dictionary of schemas
    Args: 
        config_path (str): path to the config file
    Returns:
        schemas (dict)
    """
    config = configparser.ConfigParser()
    config.read(config_path)

    schemas = {}
    for section in config.sections():
        schemas[section] = dict(config.items(section))
    return schemas

In [0]:
# Call the 000-create-schema in a loop
load_all_schemas = load_schemas_config(config_path=config_path_file)
logger.info("Schemas Available")

section_key = "schema_names"  # config file uses [schema_names]
if section_key in load_all_schemas:
    schemas = load_all_schemas[section_key]
    logger.info(f"\nSchemas to be created in env:")
else:
    raise logger.error(f"Section {section_key} Not found")

# notebook caller 000-create-schema

env = 'dev'
for schema_key, schema_name in schemas.items():
    logger.info(f"\nCreating Schema: {schema_name} (Environment: {env})")
    try:
        result = dbutils.notebook.run(
            path = notebook_path,
            timeout_seconds=600,
            arguments={
                "env":env,
                "schema_name":schema_name
            }
        )
        logger.info(f"Schema {schema_name} created successfully")
    except Exception as e:
        logger.error(f"Error creating schema {schema_name}: {e}")
        raise

    